In [1]:
import pandas as pd
import polars as pl
from datetime import datetime, timedelta,date
import numpy as np
from pathlib import Path

In [2]:
data = pd.DataFrame() 
years = ["2022","2023","2024","2025"]

for y in years:
    PATH = f"../data/processed/statcast_{y}.parquet"
    data = pd.concat([data, pd.read_parquet(PATH)])

In [3]:
data = pl.DataFrame(data)

In [4]:
# data.columns

In [5]:
#missed balls
data = (
    data
    .with_columns (
        (
            (
                (pl.col("plate_x") < -0.7803)
                | (pl.col("plate_x") > 0.7803)

                | (pl.col("plate_z") < pl.col("sz_bot"))
                | (pl.col("plate_z") > pl.col("sz_top"))
            )
           & (pl.col("description")=="called_strike")
        )
        
         .cast(pl.Int8)
        .alias("is_missed_ball")
    )
)

In [6]:
#missed strikes
data = (
    data
    .with_columns (
        (
            (
                (pl.col("plate_x") >= -0.7803)
                & (pl.col("plate_x") <= 0.7803)

                & (pl.col("plate_z") >= pl.col("sz_bot"))
                & (pl.col("plate_z") <= pl.col("sz_top"))
            )
           & (pl.col("description")=="ball")
        )
        
         .cast(pl.Int8)
        .alias("is_missed_strike")
    )
)

In [7]:
# # source:
# #https://tangotiger.com/index.php/site/article/statcast-lab-swing-take-and-a-primer-on-run-value

# def add_shadow_zones(df,plate_in = 8.5, name = "attack_zone"):
#     """
#     Heart  = inside the rule-book zone
#     Shadow = within one ball-width of the zone edge
#     Chase  = within `ball_in + waste_in` of the zone edge
#     Waste  = beyond that
#     """
#     required = ["plate_x", "plate_z", "sz_top", "sz_bot"]
#     missing = pl.any_horizontal(*[pl.col(c).is_null() for c in required])

#     # df = df.lazy()
#     # output
#     output = (
#         df.with_columns(
#             sz_mid = (pl.col("sz_top") + pl.col("sz_bot")) / 2,
#             sz_half = ((pl.col("sz_top") - pl.col("sz_bot")) / 2) * 12,
#         )
#         .with_columns(
#             dx = pl.col("plate_x").abs() * 12 - plate_in,
#             dz = (pl.col("plate_z") - pl.col("sz_mid")).abs() * 12 - pl.col("sz_half"),
#         )
#         .with_columns(
#             dist = pl.max_horizontal(pl.col("dx"), pl.col("dz"))
#         )
#          .with_columns(
#             pl.when(missing)
#               .then(pl.lit(None, dtype=pl.Utf8))
#               .when(pl.col("dist") <= -2.9).then(pl.lit("Heart"))
#               .when(pl.col("dist") <= 0).then(pl.lit("Shadow"))
#               .when(pl.col("dist") <= 1.5).then(pl.lit("Shadow"))
#               .when(pl.col("dist") <= 8.5).then(pl.lit("Chase"))
#               .otherwise(pl.lit("Waste"))
#               .alias(name)
#         )
#         .drop(["sz_mid","sz_half","dx","dz","dist"])
#     )
#     return output
# data = add_shadow_zones(data)

In [8]:
data = data.filter(pl.col("description").is_in(["called_strike", "ball"]) & 
                (pl.col("plate_x").is_not_null()) &
                (pl.col("sz_top").is_not_null())
                )
data = data.with_columns((pl.col("is_missed_ball") | pl.col("is_missed_strike")).alias("wrong_call"))


In [9]:
x_bins = [-0.915, -0.28, 0.28, 0.915]
z_bins = [-0.0125, 1/3, 2/3, 1.0125]

In [10]:
data = (
    data
    .with_columns
    (
        pl.when(pl.col("stand")=="R").then(pl.col("plate_x"))
        .otherwise(-1*pl.col("plate_x"))
        .alias("plate_x_bat_norm")
    )
    .with_columns
    (
        (
            (pl.col("plate_z") - pl.col("sz_bot"))
            /(pl.col("sz_top") - pl.col("sz_bot"))
        ).alias("z_norm")
    )
)

In [11]:
data = (
    data
    .with_columns
    (
        x_bin=pl.col("plate_x_bat_norm")
        .bin_intervals
        (
            intervals=x_bins,
            labels = False
        )
    
    )
    .with_columns
    (
        z_bin=pl.col("z_norm")
        .bin_intervals
        (
            intervals=z_bins,
            labels = False
        )
    
    )
    .with_columns
    (
        ((pl.col("x_bin") + pl.col("z_bin")*5))
        .alias("cell")
    
    )
    .with_columns
    (
        (pl.col("cell")>19)
        .alias("High")
    )
    .with_columns
    (
        (pl.col("cell")<5)
        .alias("Low")
    )
    .with_columns
    (
        (pl.col("cell").is_in([0,5,10,15,20]))
        .alias("Inside")
    )
    .with_columns
    (
        (pl.col("cell").is_in([4,9,14,19,24]))
        .alias("Outside")
    )
    .drop("x_bin","z_bin","cell")
)

In [12]:
data = data.to_pandas()

In [13]:
pitch_mapping = {
    # Fastballs (Velocity & Movement variations)
    "FF": "Fastball",  # Four-Seam Fastball
    "SI": "Fastball",  # Sinker
    "FC": "Fastball",  # Cutter
    "FA": "Fastball",  # Other Fastball

    # Sliders (Horizontal movement)
    "SL": "Slider",    # TSlider
    "ST": "Slider",    # Sweeper
    "SV": "Slider",    # Slurve

    # Curveballs
    "CU": "Curveball", # Traditional Curveball
    "KC": "Curveball", # Knuckle-Curve
    "CS": "Curveball", # Slow Curve

    # Changeups
    "CH": "Changeup",  # Traditional Changeup
    "FS": "Changeup",  # Splitter (Finger-Split)
    "FO": "Changeup",  # Forkball
    "SC": "Changeup",  # Screwball (Fades opposite of a slider)

    # Junk
    "EP": "Junk",      # Eephus (Ultra-slow utility pitch)
    "KN": "Junk"       # Knuckleball
}

data["pitch_cat"] = data["pitch_type"].map(pitch_mapping)
data["count"] = data["balls"].astype(str) + "-" + data["strikes"].astype(str)
data["same_hand"] = (data["stand"] == data["p_throws"]).astype(int)
data["wrong_call"] = data["is_missed_ball"] | data["is_missed_strike"]
data['call'] = data['description']
data["month"] = pd.to_datetime(data["game_date"]).dt.month
data["zone"] = data["zone"].astype(int).astype(str) 
data["pitcher"] = data["pitcher"].astype(int).astype(str) 
data["batter"] = data["batter"].astype(int).astype(str) 
data["called_strike"] = (data["description"] == "called_strike").astype(int)

In [14]:
#data.info(verbose=True)

In [15]:
feature_cols = [
    "release_speed",
    "release_spin_rate",
    "pitch_cat",
    #"zone",
    "balls",
    "strikes",
    "inning",
    "outs_when_up",
    "inning_topbot",
    "stand",
    "p_throws",
    "n_thruorder_pitcher",
    "sz_top",
    "sz_bot",
    "called_strike",
    "pitcher",
    "batter",
    "pitch_number",
    "pitcher_days_since_prev_game",
    #"attack_zone",
    "hp_official_id",
    "count",
    "High",
    "Low",
    "Inside",
    "Outside"
    #"cum_ump_miss_rate"

]

response_cols = [
    "is_missed_ball",
    "is_missed_strike",
    "wrong_call"
    ]

In [16]:
# data.describe()

In [17]:
train_dir = Path("../data/processed/train")
train_dir.mkdir(exist_ok=True)

test_dir = Path("../data/processed/test")
test_dir.mkdir(exist_ok=True)

features_dir = Path("../data/processed/features")
features_dir.mkdir(parents=True,exist_ok=True)

response_cols_dir = Path("../data/processed/response_cols")
response_cols_dir.mkdir(exist_ok=True)

In [18]:
pd.Series(feature_cols, name="feature").to_csv(
    features_dir / "feature_cols.csv",
    index=False
)

pd.Series(response_cols, name="response").to_csv(
    response_cols_dir / "response_cols.csv",
    index=False
)

In [19]:
test_data = data[data["game_date"].dt.date > date(2025, 1, 1)]
train_data = data.drop(test_data.index)
test_data , train_data  = test_data.reset_index(drop=True), train_data.reset_index(drop=True)

In [20]:
train_path = train_dir / f"train_data.parquet"
train_data.to_parquet(train_path,index=False)

test_path = test_dir / f"test_data.parquet"
test_data.to_parquet(test_path,index=False)

In [21]:
train_data.head(25)

,pitch_type,game_date,release_speed,release_pos_x,release_pos_z,player_name,batter,pitcher,events,description,...,High,Low,Inside,Outside,pitch_cat,count,same_hand,call,month,called_strike
0,FF,2022-04-03,94.5,-0.94,5.51,"Duarte, Daniel",647378,650960,None,called_strike,...,False,False,False,False,Fastball,1-0,0,called_strike,4,1
1,FF,2022-04-03,95.6,-1.13,5.65,"Duarte, Daniel",647378,650960,None,ball,...,False,False,False,False,Fastball,0-0,0,ball,4,0
2,FF,2022-04-03,94.4,-1.27,5.52,"Duarte, Daniel",623323,650960,walk,ball,...,False,False,True,False,Fastball,3-0,1,ball,4,0
3,FF,2022-04-03,95.5,-1.02,5.75,"Duarte, Daniel",623323,650960,None,ball,...,False,False,False,False,Fastball,2-0,1,ball,4,0
4,ST,2022-04-03,85.9,-1.20,5.55,"Duarte, Daniel",678489,650960,None,ball,...,False,True,True,False,Slider,0-1,0,ball,4,0
5,FF,2022-04-03,94.7,-0.82,5.63,"Duarte, Daniel",678489,650960,None,called_strike,...,False,True,False,False,Fastball,0-0,0,called_strike,4,1
6,CH,2022-04-03,89.2,-1.14,5.39,"Duarte, Daniel",669450,650960,None,ball,...,False,True,False,False,Changeup,1-1,0,ball,4,0
7,ST,2022-04-03,85.1,-1.09,5.54,"Duarte, Daniel",645444,650960,None,ball,...,False,False,True,False,Slider,1-1,0,ball,4,0
8,FF,2022-04-03,94.6,-0.98,5.67,"Duarte, Daniel",645444,650960,None,called_strike,...,False,False,False,False,Fastball,1-0,0,called_strike,4,1
9,CH,2022-04-03,91.0,-1.12,5.54,"Duarte, Daniel",645444,650960,None,ball,...,False,True,False,False,Changeup,0-0,0,ball,4,0
